In [1]:
import pandas as pd
import numpy as np

Loading the Dataset

In [ ]:
train_data = pd.read_csv("../data/train.csv")
test_data = pd.read_csv("../data/test.csv")
sample_submission = pd.read_csv("../data/sample_submission.csv")

Inspecting the Dataset

In [3]:
print("Training data shape:", train_data.shape)
print("Test data shape:", test_data.shape)
print("Sample submission shape:", sample_submission.shape)

Training data shape: (594194, 21)
Test data shape: (254655, 20)
Sample submission shape: (254655, 2)


In [4]:
train_data.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
test_data.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [6]:
sample_submission.head()

,id,Churn
0,594194,0
1,594195,0
2,594196,0
3,594197,0
4,594198,0


In [7]:
train_data.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [8]:
train_data.dtypes

id                    int64
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

Missing Values

In [9]:
train_missing = train_data.isnull().sum()
test_missing = test_data.isnull().sum()

print("Missing values in training data:")
print(train_missing[train_missing > 0])

print("\nMissing values in test data:")
print(test_missing[test_missing > 0])

Missing values in training data:
Series([], dtype: int64)

Missing values in test data:
Series([], dtype: int64)


In [10]:
# Check duplicate records and identifiers
print("Duplicate training rows:", train_data.duplicated().sum())
print("Duplicate test rows:", test_data.duplicated().sum())

print("Duplicate training IDs:", train_data["id"].duplicated().sum())
print("Duplicate test IDs:", test_data["id"].duplicated().sum())

print("Unique training IDs:", train_data["id"].nunique())
print("Unique test IDs:", test_data["id"].nunique())

Duplicate training rows: 0
Duplicate test rows: 0
Duplicate training IDs: 0
Duplicate test IDs: 0
Unique training IDs: 594194
Unique test IDs: 254655


Check Target Distribution

In [11]:
churn_counts = train_data["Churn"].value_counts()
churn_percentages = train_data["Churn"].value_counts(normalize=True) * 100

print(churn_counts)
print("\nChurn percentage:")
print(churn_percentages.round(2))

Churn
No     460377
Yes    133817
Name: count, dtype: int64

Churn percentage:
Churn
No     77.48
Yes    22.52
Name: proportion, dtype: float64


Seperating Numerical and Categorical Columns

In [12]:
# Separate the target and identifier from the model features
identifier_column = "id"
target_column = "Churn"

feature_data = train_data.drop(
    columns=[identifier_column, target_column]
)

numerical_columns = (
    feature_data
    .select_dtypes(include=["int64", "float64"])
    .columns
    .tolist()
)

categorical_columns = (
    feature_data
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("Number of model features:", feature_data.shape[1])
print("Number of numerical features:", len(numerical_columns))
print("Number of categorical features:", len(categorical_columns))

print("\nNumerical features:")
print(numerical_columns)

print("\nCategorical features:")
print(categorical_columns)

Number of model features: 19
Number of numerical features: 4
Number of categorical features: 15

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
